# Algorithm 2: Morphology-guided Watershed

Pure computer-vision segmentation — **no deep learning, no training required**.

## Pipeline
```
Image
  → Bilateral filter  (noise reduction, edge preservation)
  → Adaptive threshold (Gaussian, grayscale)  →  binary foreground mask
  → Morphological Open + Close  (clean binary mask)
  → Morphological Erosion × 3  →  sure-foreground seeds  (shape-guided)
  → Morphological Dilation × 3  →  sure-background
  → Connected components → Watershed markers
  → cv2.watershed()  →  instance label map
  → Extract each label as a binary mask (filter by min area)
```

**Key difference from Algorithm 1:** Seeds are determined by erosion depth rather than
distance-transform peaks. Erosion shrinks every object inward uniformly, so seeds reflect
the *shape* of the object. Better for elongated or irregular garbage items (bags, strips).

**Run after:** `remap_classes.py` (annotations must already be remapped to 3 classes).

In [ ]:
import cv2
import numpy as np
import json
import os
import time
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

BASE_DIR    = r"C:\Users\User\Desktop\Ipynb"
DATASET_DIR = os.path.join(BASE_DIR, "archive", "dataset_v2")
OUTPUT_DIR  = os.path.join(BASE_DIR, "runs", "watershed_morphology")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MIN_AREA    = 500   # pixels — smaller segments are treated as noise
CLASS_NAMES = ["Recyclable", "Non-recyclable", "Other"]

print(f"Output directory: {OUTPUT_DIR}")

## Core Algorithm

In [ ]:
def morphology_watershed_segments(image, min_area=500):
    """
    Segment objects in a full image using morphological-erosion seeding.
    Returns a list of binary masks (uint8, 0/255), one per detected segment.
    """
    h, w = image.shape[:2]

    # ── Preprocessing ────────────────────────────────────────────────────────
    filtered = cv2.bilateralFilter(image, 9, 75, 75)
    gray     = cv2.cvtColor(filtered, cv2.COLOR_BGR2GRAY)

    # Adaptive threshold on grayscale (inverted: objects are bright in TACO)
    binary = cv2.adaptiveThreshold(gray, 255,
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 15, 2)

    # ── Morphological cleanup ─────────────────────────────────────────────────
    kernel_s = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_m = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    binary   = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel_s, iterations=2)
    binary   = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_s, iterations=2)

    # ── Morphological erosion → sure foreground (shape-guided seeds) ──────────
    sure_fg = cv2.erode(binary, kernel_m, iterations=3)

    # ── Morphological dilation → sure background ──────────────────────────────
    sure_bg = cv2.dilate(binary, kernel_m, iterations=3)
    unknown = cv2.subtract(sure_bg, sure_fg)

    # ── Watershed markers ────────────────────────────────────────────────────
    _, markers = cv2.connectedComponents(sure_fg)
    markers    = markers + 1          # 0 → unknown; background becomes 1
    markers[unknown == 255] = 0
    markers    = cv2.watershed(image.copy(), markers)

    # ── Extract instance masks ───────────────────────────────────────────────
    masks = []
    for label in set(np.unique(markers)) - {1, -1}:   # 1=bg, -1=boundary
        m = np.zeros((h, w), dtype=np.uint8)
        m[markers == label] = 255
        if cv2.countNonZero(m) >= min_area:
            masks.append(m)

    return masks


def build_gt_mask(anns, img_shape):
    """Combine all GT annotation polygons into one binary mask."""
    h, w = img_shape[:2]
    combined = np.zeros((h, w), dtype=np.uint8)
    for ann in anns:
        for seg in ann.get("segmentation", []):
            pts = np.array(seg, dtype=np.float32).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(combined, [pts], 255)
    return combined


print("Core functions defined.")

## Batch Inference & Evaluation

In [ ]:
# ── Load test annotations ────────────────────────────────────────────────────
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    test_coco_data = json.load(f)

test_image_dir  = os.path.join(DATASET_DIR, "images", "test")
img_file_lookup = {img["file_name"]: img for img in test_coco_data["images"]}
test_files      = sorted([
    f for f in os.listdir(test_image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

gt_masks_by_image  = defaultdict(list)
gt_counts_by_image = defaultdict(int)
for ann in test_coco_data["annotations"]:
    gt_masks_by_image[ann["image_id"]].append(ann)
    gt_counts_by_image[ann["image_id"]] += 1

print(f"Test images: {len(test_files)}")
cats = {c["id"]: c["name"] for c in test_coco_data["categories"]}
print(f"Categories : {cats}")

# ── Batch inference ──────────────────────────────────────────────────────────
binary_ious     = []
count_errors    = []
inference_times = []
per_image_results = []

for fname in tqdm(test_files, desc="Morphology Watershed"):
    image = cv2.imread(os.path.join(test_image_dir, fname))
    if image is None:
        continue
    img_info = img_file_lookup.get(fname)
    if img_info is None:
        continue
    image_id = img_info["id"]

    t0    = time.perf_counter()
    masks = morphology_watershed_segments(image, min_area=MIN_AREA)
    inf_ms = (time.perf_counter() - t0) * 1000
    inference_times.append(inf_ms)

    # Binary IoU
    h, w = image.shape[:2]
    pred_combined = np.zeros((h, w), dtype=np.uint8)
    for m in masks:
        pred_combined = cv2.bitwise_or(pred_combined, m)

    gt_combined = build_gt_mask(gt_masks_by_image.get(image_id, []), (h, w))
    inter = cv2.countNonZero(cv2.bitwise_and(pred_combined, gt_combined))
    union = cv2.countNonZero(cv2.bitwise_or(pred_combined, gt_combined))
    binary_ious.append(inter / max(union, 1))

    # Counting
    gt_count   = gt_counts_by_image.get(image_id, 0)
    pred_count = len(masks)
    count_errors.append(abs(pred_count - gt_count))

    per_image_results.append({
        "file":        fname,
        "gt_count":    gt_count,
        "pred_count":  pred_count,
        "count_error": abs(pred_count - gt_count),
        "inference_ms": inf_ms,
    })

# ── Metrics ──────────────────────────────────────────────────────────────────
n        = len(count_errors)
avg_iou  = float(np.mean(binary_ious))
mae      = float(np.mean(count_errors))
within_1 = sum(1 for e in count_errors if e <= 1) / n * 100
within_3 = sum(1 for e in count_errors if e <= 3) / n * 100
avg_inf  = float(np.mean(inference_times[1:]) if len(inference_times) > 1 else inference_times[0])

print(f"\n{'='*45}")
print("Algorithm 2: Morphology-guided Watershed")
print(f"{'='*45}")
print(f"  Binary IoU:        {avg_iou:.4f}")
print(f"  Counting MAE:      {mae:.2f}")
print(f"  Within +/-1:       {within_1:.1f}%")
print(f"  Within +/-3:       {within_3:.1f}%")
print(f"  Avg inference ms:  {avg_inf:.1f}")
print(f"{'='*45}")

# ── Save results ─────────────────────────────────────────────────────────────
results_data = {
    "metrics": {
        "box_map50":         0.0,
        "box_map50_95":      0.0,
        "mask_map50":        0.0,
        "mask_map50_95":     0.0,
        "binary_iou":        avg_iou,
        "counting_mae":      mae,
        "counting_within_1": within_1,
        "counting_within_3": within_3,
        "avg_inference_ms":  avg_inf,
    },
    "per_image": per_image_results,
}

out_path = os.path.join(OUTPUT_DIR, "watershed_morphology_results.json")
with open(out_path, "w") as f:
    json.dump(results_data, f, indent=2)
print(f"\nSaved: {out_path}")

## Visualization

In [ ]:
CMAP = plt.cm.tab10.colors

samples = [f for f in test_files if cv2.imread(os.path.join(test_image_dir, f)) is not None][:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, fname in zip(axes.flat, samples):
    image     = cv2.imread(os.path.join(test_image_dir, fname))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    masks     = morphology_watershed_segments(image, min_area=MIN_AREA)
    overlay   = image_rgb.copy()

    for i, m in enumerate(masks):
        c = [int(x * 255) for x in CMAP[i % len(CMAP)][:3]]
        colored           = np.zeros_like(overlay)
        colored[m == 255] = c
        overlay = cv2.addWeighted(overlay, 1.0, colored, 0.4, 0)
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, cnts, -1, c, 2)

    img_id   = img_file_lookup.get(fname, {}).get("id", -1)
    gt_count = gt_counts_by_image.get(img_id, 0)
    ax.imshow(overlay)
    ax.set_title(f"GT: {gt_count} | Pred: {len(masks)}", fontsize=10)
    ax.axis("off")

plt.suptitle("Algorithm 2: Morphology-guided Watershed", fontsize=14, fontweight="bold")
plt.tight_layout()
viz_path = os.path.join(OUTPUT_DIR, "watershed_morphology_predictions.png")
plt.savefig(viz_path, dpi=150)
plt.show()
print(f"Saved: {viz_path}")